# 4b. cVAE Batch Correction — Improved (z-score + latent_dim sweep)

**Improvements over notebook 4:**
- ✅ Z-score normalization of input embeddings
- ✅ Latent dim sweep: 32, 64, 128
- ✅ Results saved to `data/processed_up/`
- ✅ Final comparison with original `data/processed/` results

For each `latent_dim`, corrected embeddings are stored as:
```
FM_COMPASS_PT_embedding_scVI_corrected_d32
FM_COMPASS_PT_embedding_scVI_corrected_d64
FM_COMPASS_PT_embedding_scVI_corrected_d128
```

## 0. Setup

In [ ]:
!git clone -b feat/scvi-batch-correction https://github.com/onion-42/batchcor-rna-embeds.git 2>/dev/null || (cd batchcor-rna-embeds && git pull origin feat/scvi-batch-correction)
%cd batchcor-rna-embeds

!pip install -q uv
!uv pip install --system -e .
!pip install -q --upgrade ipython ipykernel

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from pathlib import Path
from loguru import logger
from matplotlib.lines import Line2D
from sklearn.metrics import silhouette_score
from umap import UMAP

from batchcor_rna_emb.logging_config import set_logger
from batchcor_rna_emb.config import (
    BATCH_COL, DIAGNOSIS_COL, SEED,
    COMPASS_PT_EMBEDDING_KEY,
    SCVI_CORRECTED_KEY,
    COHORT_CANCER_CODE,
)
from batchcor_rna_emb.data_io import load_cohort, save_adata_zarr
from batchcor_rna_emb.split_utils import add_dummy_split, get_split_masks
from batchcor_rna_emb.batch_correction.scvi_corrector import CVAECorrector, CVAEConfig

set_logger(level="INFO")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
logger.info("Device: {}, PyTorch: {}", DEVICE, torch.__version__)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DATA_INTERIM_PT_DIR = Path('/content/drive/MyDrive/data/interim_PT')
DATA_PROCESSED_DIR  = Path('/content/drive/MyDrive/data/processed')       # old results
DATA_PROCESSED_UP   = Path('/content/drive/MyDrive/data/processed_up')    # new results
DATA_PROCESSED_UP.mkdir(parents=True, exist_ok=True)

SPLIT_COL = "Split_dummy"
LATENT_DIMS = [32, 64, 128]

logger.info("Source: {}", [p.name for p in sorted(DATA_INTERIM_PT_DIR.glob('*.adata.zarr'))])

## 1. Plot Helper

In [ ]:
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.facecolor": "#f8f9fa",
    "axes.facecolor":   "#ffffff",
    "figure.dpi":       120,
})


def plot_umap_panel(coords, labels, title, ax, palette=None):
    unique = sorted(labels.unique())
    if palette is None:
        palette = sns.color_palette("husl", n_colors=len(unique))
    cmap = dict(zip(unique, palette))
    colors = [cmap[l] for l in labels]
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, s=10, alpha=0.6,
               edgecolors="none", rasterized=True)
    ax.set_title(title, fontsize=9, fontweight="bold")
    if len(unique) <= 12:
        handles = [Line2D([0], [0], marker='o', color='w',
                   markerfacecolor=cmap[l], markersize=6, label=str(l))
                   for l in unique]
        ax.legend(handles=handles, fontsize=5, loc="best",
                  framealpha=0.8, edgecolor="#ccc")

## 2. Run Latent Dim Sweep (with z-score)

For each cohort × each latent_dim (32, 64, 128):
- Z-score normalize input
- Fit cVAE on train
- Transform all
- Store as separate `.obsm` keys

In [ ]:
all_histories = {}  # (cohort, latent_dim) -> history

for cohort_name in COHORT_CANCER_CODE:
    src_path  = DATA_INTERIM_PT_DIR / f"{cohort_name}.adata.zarr"
    dest_path = DATA_PROCESSED_UP   / f"{cohort_name}.adata.zarr"

    if not src_path.exists():
        logger.warning("Missing in interim_PT: {}", cohort_name)
        continue

    if dest_path.exists():
        logger.info("Already in processed_up, skipping: {}", cohort_name)
        continue

    logger.info("\n" + "=" * 70)
    logger.info("Cohort: {}", cohort_name)
    logger.info("=" * 70)

    adata = load_cohort(src_path)

    if COMPASS_PT_EMBEDDING_KEY not in adata.obsm:
        logger.error("Missing '{}', skipping", COMPASS_PT_EMBEDDING_KEY)
        del adata
        continue

    adata = add_dummy_split(adata, col_name=SPLIT_COL, seed=SEED)
    train_mask, test_mask = get_split_masks(adata, SPLIT_COL)

    emb_all = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    batch_labels = adata.obs[BATCH_COL].astype(str).values

    logger.info("Samples: {}, Batches: {}, Train: {}, Test: {}",
        adata.n_obs, len(set(batch_labels)),
        train_mask.sum(), test_mask.sum(),
    )

    for ldim in LATENT_DIMS:
        logger.info("--- latent_dim={} ---", ldim)

        cfg = CVAEConfig(
            latent_dim=ldim,
            hidden_dims=(512, 256),
            n_epochs=150,
            batch_size=128,
            lr=1e-3,
            beta_max=1.0,
            warmup_fraction=0.3,
            dropout=0.2,
            normalize=True,    # <-- z-score enabled
            seed=SEED,
        )

        corrector = CVAECorrector(config=cfg)
        corrector.fit(X=emb_all[train_mask], batch_labels=batch_labels[train_mask])
        corrected = corrector.transform(emb_all)

        key = f"{SCVI_CORRECTED_KEY}_d{ldim}"
        adata.obsm[key] = corrected
        all_histories[(cohort_name, ldim)] = corrector.history_

        logger.info("Stored '{}' shape={}", key, corrected.shape)
        assert np.isfinite(corrected).all()

    # Save
    save_adata_zarr(adata, dest_path)
    logger.info("Saved: {}", dest_path)

    del adata
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

logger.info("\nAll cohorts processed.")

## 3. Training Curves — All Latent Dims

In [ ]:
cohort_names = sorted(set(k[0] for k in all_histories))
if not cohort_names:
    logger.warning("No histories to plot.")
else:
    for cname in cohort_names:
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(f"{cname} — Training Curves (z-score ON)",
                     fontsize=13, fontweight="bold", y=1.03)

        colors = {32: '#E91E63', 64: '#FF9800', 128: '#2196F3'}

        for ldim in LATENT_DIMS:
            hist = all_histories.get((cname, ldim))
            if hist is None:
                continue
            epochs = range(1, len(hist.total) + 1)
            c = colors[ldim]
            axes[0].plot(epochs, hist.recon, color=c, lw=1.5, label=f"d={ldim}")
            axes[1].plot(epochs, hist.kl,    color=c, lw=1.5, label=f"d={ldim}")
            axes[2].plot(epochs, hist.total,  color=c, lw=1.5, label=f"d={ldim}")

        axes[0].set_title("Reconstruction (MSE)"); axes[0].legend(fontsize=8)
        axes[1].set_title("KL Divergence");        axes[1].legend(fontsize=8)
        axes[2].set_title("Total Loss");            axes[2].legend(fontsize=8)
        for ax in axes:
            ax.set_xlabel("Epoch")

        plt.tight_layout()
        plt.show()

## 4. UMAP — Before vs After (per latent_dim)

For the first available multi-batch cohort.

In [ ]:
# Find first multi-batch cohort
test_cohort = None
for name in COHORT_CANCER_CODE:
    p = DATA_PROCESSED_UP / f"{name}.adata.zarr"
    if p.exists():
        ad_tmp = load_cohort(p)
        if ad_tmp.obs[BATCH_COL].nunique() > 1:
            test_cohort = name
            del ad_tmp
            break
        del ad_tmp

if test_cohort:
    adata = load_cohort(DATA_PROCESSED_UP / f"{test_cohort}.adata.zarr")
    batch = adata.obs[BATCH_COL].astype(str)
    diag  = adata.obs[DIAGNOSIS_COL].astype(str)
    batch_pal = sns.color_palette("husl", n_colors=batch.nunique())
    diag_pal  = sns.color_palette("Set2", n_colors=diag.nunique())

    n_cols = 1 + len(LATENT_DIMS)  # before + each latent_dim
    fig, axes = plt.subplots(2, n_cols, figsize=(5 * n_cols, 10))
    fig.suptitle(
        f"{test_cohort} — Before vs After (z-score + latent_dim sweep)",
        fontsize=14, fontweight="bold", y=1.02,
    )

    # Before UMAP
    emb_raw = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    umap_raw = UMAP(n_components=2, random_state=SEED).fit_transform(emb_raw)
    plot_umap_panel(umap_raw, batch, "Before — Batch",     axes[0, 0], batch_pal)
    plot_umap_panel(umap_raw, diag,  "Before — Diagnosis", axes[1, 0], diag_pal)

    # After UMAP for each latent_dim
    for col_idx, ldim in enumerate(LATENT_DIMS, start=1):
        key = f"{SCVI_CORRECTED_KEY}_d{ldim}"
        if key not in adata.obsm:
            continue
        emb_corr = np.asarray(adata.obsm[key], dtype=np.float32)
        umap_corr = UMAP(n_components=2, random_state=SEED).fit_transform(emb_corr)
        plot_umap_panel(umap_corr, batch, f"d={ldim} — Batch",     axes[0, col_idx], batch_pal)
        plot_umap_panel(umap_corr, diag,  f"d={ldim} — Diagnosis", axes[1, col_idx], diag_pal)

    for ax in axes.flat:
        ax.set_xlabel("UMAP-1"); ax.set_ylabel("UMAP-2")
    plt.tight_layout()
    plt.show()
    del adata
else:
    logger.warning("No multi-batch cohort found.")

## 5. Silhouette Scores — All Latent Dims

In [ ]:
rows_up = []

for cohort_name in COHORT_CANCER_CODE:
    p = DATA_PROCESSED_UP / f"{cohort_name}.adata.zarr"
    if not p.exists():
        continue

    adata = load_cohort(p)
    emb_raw = np.asarray(adata.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    batch = adata.obs[BATCH_COL].astype(str).values
    diag  = adata.obs[DIAGNOSIS_COL].astype(str).values

    multi_batch = len(set(batch)) > 1
    multi_diag  = len(set(diag))  > 1

    sil_b_raw = silhouette_score(emb_raw, batch) if multi_batch else np.nan
    sil_d_raw = silhouette_score(emb_raw, diag)  if multi_diag  else np.nan

    for ldim in LATENT_DIMS:
        key = f"{SCVI_CORRECTED_KEY}_d{ldim}"
        if key not in adata.obsm:
            continue
        emb_corr = np.asarray(adata.obsm[key], dtype=np.float32)

        sil_b = silhouette_score(emb_corr, batch) if multi_batch else np.nan
        sil_d = silhouette_score(emb_corr, diag)  if multi_diag  else np.nan

        rows_up.append({
            "Cohort": cohort_name,
            "latent_dim": ldim,
            "Sil_Batch_Before": round(sil_b_raw, 4),
            "Sil_Batch_After":  round(sil_b, 4),
            "Sil_Batch_Δ":      round(sil_b - sil_b_raw, 4) if not np.isnan(sil_b) else np.nan,
            "Sil_Diag_Before":  round(sil_d_raw, 4),
            "Sil_Diag_After":   round(sil_d, 4),
            "Sil_Diag_Δ":       round(sil_d - sil_d_raw, 4) if not np.isnan(sil_d) else np.nan,
        })

    del adata

df_up = pd.DataFrame(rows_up)
display(df_up)

logger.info("Sil_Batch_Δ < 0 = better mixing (good)")
logger.info("Sil_Diag_Δ >= 0 = biology preserved (good)")

## 6. Comparison: Original (no z-score, d=128) vs Improved

Side-by-side comparison of `data/processed/` (old) vs `data/processed_up/` (new).

In [ ]:
rows_cmp = []

for cohort_name in COHORT_CANCER_CODE:
    old_path = DATA_PROCESSED_DIR / f"{cohort_name}.adata.zarr"
    new_path = DATA_PROCESSED_UP  / f"{cohort_name}.adata.zarr"

    if not old_path.exists() or not new_path.exists():
        continue

    adata_old = load_cohort(old_path)
    adata_new = load_cohort(new_path)

    batch = adata_old.obs[BATCH_COL].astype(str).values
    diag  = adata_old.obs[DIAGNOSIS_COL].astype(str).values
    multi_batch = len(set(batch)) > 1
    multi_diag  = len(set(diag))  > 1

    # Raw (uncorrected)
    emb_raw = np.asarray(adata_old.obsm[COMPASS_PT_EMBEDDING_KEY], dtype=np.float32)
    sil_b_raw = silhouette_score(emb_raw, batch) if multi_batch else np.nan
    sil_d_raw = silhouette_score(emb_raw, diag)  if multi_diag  else np.nan

    # Old: no z-score, d=128
    if SCVI_CORRECTED_KEY in adata_old.obsm:
        emb_old = np.asarray(adata_old.obsm[SCVI_CORRECTED_KEY], dtype=np.float32)
        sil_b_old = silhouette_score(emb_old, batch) if multi_batch else np.nan
        sil_d_old = silhouette_score(emb_old, diag)  if multi_diag  else np.nan
    else:
        sil_b_old = sil_d_old = np.nan

    # New: z-score + each latent_dim
    for ldim in LATENT_DIMS:
        key = f"{SCVI_CORRECTED_KEY}_d{ldim}"
        if key not in adata_new.obsm:
            continue
        emb_new = np.asarray(adata_new.obsm[key], dtype=np.float32)
        sil_b_new = silhouette_score(emb_new, batch) if multi_batch else np.nan
        sil_d_new = silhouette_score(emb_new, diag)  if multi_diag  else np.nan

        rows_cmp.append({
            "Cohort": cohort_name,
            "Variant": f"z-score + d={ldim}",
            "Sil_Batch_Raw":   round(sil_b_raw, 4),
            "Sil_Batch_Old":   round(sil_b_old, 4),
            "Sil_Batch_New":   round(sil_b_new, 4),
            "Old_vs_Raw":      round(sil_b_old - sil_b_raw, 4) if not np.isnan(sil_b_old) else np.nan,
            "New_vs_Raw":      round(sil_b_new - sil_b_raw, 4) if not np.isnan(sil_b_new) else np.nan,
            "New_vs_Old":      round(sil_b_new - sil_b_old, 4) if not (np.isnan(sil_b_new) or np.isnan(sil_b_old)) else np.nan,
            "Sil_Diag_Raw":   round(sil_d_raw, 4),
            "Sil_Diag_Old":   round(sil_d_old, 4),
            "Sil_Diag_New":   round(sil_d_new, 4),
        })

    del adata_old, adata_new

if rows_cmp:
    df_cmp = pd.DataFrame(rows_cmp)
    display(df_cmp)

    print("\n--- Summary ---")
    print("New_vs_Old < 0  → new is BETTER at batch mixing")
    print("New_vs_Old > 0  → old was better")
    print("\nBest variant per cohort (lowest Sil_Batch_New):")
    best = df_cmp.loc[df_cmp.groupby('Cohort')['Sil_Batch_New'].idxmin()]
    display(best[['Cohort', 'Variant', 'Sil_Batch_Raw', 'Sil_Batch_Old', 'Sil_Batch_New', 'Sil_Diag_New']])
else:
    logger.warning("No cohorts found in both processed/ and processed_up/.")